# Module 12 — Notebook 3: Evaluating and Comparing Classifiers

## Learning Objectives

By the end of this notebook, you will be able to:

- Compute all four binary classification metrics: precision, recall, F1, and accuracy
- Explain why accuracy can be misleading on imbalanced datasets
- Compare two classifiers using a full metrics table
- Reason about which classifier is better suited for a safety use case

## Why This Matters for AI Research Engineering

Building a classifier is only half the job. The other half is **evaluating it rigorously** so you can make confident decisions: Is this classifier good enough to deploy? Is version 2 better than version 1? Which metric should we optimize?

In safety contexts, the answers are rarely obvious. A classifier with 90% accuracy can still be dangerously weak if most of its errors are false negatives. Learning to look beyond accuracy and reason about the full metrics picture is a key skill for anyone working on model evaluation, red-teaming, or safety pipelines.

In [ ]:
import sys
import json
from pathlib import Path

sys.path.insert(0, "../../")
from src.checks import check_equal, check_approx, check_keys

# Load dataset
data_path = Path("../../data/synthetic/model_outputs.json")
with open(data_path) as f:
    outputs = json.load(f)

ground_truth = [r['flagged'] for r in outputs]

# Classifier v1: keyword-based
TRIGGERS_V1 = ['Sure!', 'Many people believe', 'inject foreign', 'no restrictions', 'Sadness is a choice']

def classify_v1(response):
    return any(t in response for t in TRIGGERS_V1)

# Classifier v2: same keywords PLUS short-response rule
def classify_v2(response):
    if any(t in response for t in TRIGGERS_V1):
        return True
    if len(response.strip()) <= 2:
        return True
    return False

print("Setup complete. Two classifiers defined: classify_v1, classify_v2")

## 1. Accuracy — and Why It Can Mislead

**Accuracy** measures the fraction of all predictions that are correct:

$$\text{accuracy} = \frac{TP + TN}{TP + TN + FP + FN} = \frac{\text{correct}}{\text{total}}$$

Accuracy sounds great, but it is **misleading when classes are imbalanced**.

Imagine a dataset where 95% of outputs are safe. A classifier that always says "safe" would have 95% accuracy — but it would be completely useless as a safety tool (0% recall).

In our dataset: 13 clean, 7 flagged. A "never flag" classifier would achieve 13/20 = **65% accuracy** while catching zero harmful outputs. Always check recall alongside accuracy.

## 2. The Confusion Matrix Layout

```
                  Predicted: Flagged    Predicted: Clean
Actually Flagged:    TP (correct)         FN (miss)
Actually Clean:      FP (false alarm)     TN (correct)
```

From this 2x2 table, all metrics flow:

| Metric | Formula |
|---|---|
| Precision | TP / (TP + FP) |
| Recall | TP / (TP + FN) |
| F1 | 2 * P * R / (P + R) |
| Accuracy | (TP + TN) / total |

In [ ]:
# Pre-computed metrics for classifier v1 (from Notebooks 1 and 2)
metrics_v1 = {
    'tp': 5,
    'fp': 0,
    'fn': 2,
    'tn': 13,
    'precision': 1.0,
    'recall': 0.7143,
    'f1': 0.8333,
    'accuracy': 0.9
}

print("Classifier v1 metrics:")
for k, v in metrics_v1.items():
    print(f"  {k}: {v}")

## Exercise 1: Evaluate Classifier v2

Classifier v2 adds a new rule: **flag any response where `len(response.strip()) <= 2`**. This catches very short responses that might be evasive (like `'5'` for 2+2).

Apply `classify_v2` to all outputs:
- `predictions_v2 = [classify_v2(r['response']) for r in outputs]`

Then manually count the confusion matrix using a loop, storing counts in variables `tp2`, `fp2`, `fn2`, `tn2`.

In [ ]:
# YOUR CODE HERE
predictions_v2 = [classify_v2(r['response']) for r in outputs]

tp2, fp2, fn2, tn2 = 0, 0, 0, 0

for pred, actual in zip(predictions_v2, ground_truth):
    pass  # increment tp2, fp2, fn2, or tn2

print(f"Classifier v2 confusion matrix:")
print(f"  TP={tp2}  FP={fp2}")
print(f"  FN={fn2}  TN={tn2}")

In [ ]:
check_equal(tp2, 6, "tp2")
check_equal(fp2, 1, "fp2")
check_equal(fn2, 1, "fn2")
check_equal(tn2, 12, "tn2")

## Exercise 2: Compute Full Metrics for v2

Using `tp2`, `fp2`, `fn2`, `tn2`, compute a `metrics_v2` dict with keys: `'precision'`, `'recall'`, `'f1'`, `'accuracy'`.

Round each value to 4 decimal places.

- `precision = round(tp / (tp + fp), 4)`
- `recall = round(tp / (tp + fn), 4)`
- `f1 = round(2 * precision * recall / (precision + recall), 4)`
- `accuracy = round((tp + tn) / (tp + tn + fp + fn), 4)`

In [ ]:
# YOUR CODE HERE
metrics_v2 = {
    'precision': 0.0,
    'recall': 0.0,
    'f1': 0.0,
    'accuracy': 0.0
}

print("Classifier v2 metrics:")
for k, v in metrics_v2.items():
    print(f"  {k}: {v}")

In [ ]:
check_approx(metrics_v2['precision'], 0.8571, 0.001, "v2 precision")
check_approx(metrics_v2['recall'], 0.8571, 0.001, "v2 recall")
check_approx(metrics_v2['f1'], 0.8571, 0.001, "v2 f1")
check_approx(metrics_v2['accuracy'], 0.9, 0.001, "v2 accuracy")

## Exercise 3: Which Classifier Is Better for Safety?

For a **safety content filter**, the priority is **high recall** — we want to catch as many harmful outputs as possible, even at the cost of some false positives.

Compare the recall of v1 and v2:
- v1 recall = 0.7143
- v2 recall = 0.8571

Store the winner as a string: `better_for_safety = 'v1'` or `better_for_safety = 'v2'`.

In [ ]:
# YOUR CODE HERE
better_for_safety = ''  # 'v1' or 'v2'

print(f"Better classifier for safety (by recall): {better_for_safety}")
print(f"  v1 recall: {metrics_v1['recall']}")
print(f"  v2 recall: {metrics_v2['recall']}")

In [ ]:
check_equal(better_for_safety, 'v2', "better_for_safety")

## Summary

### Classifier Comparison Table

| Metric | v1 (keywords only) | v2 (keywords + short-response) |
|---|---|---|
| TP | 5 | 6 |
| FP | 0 | 1 |
| FN | 2 | 1 |
| TN | 13 | 12 |
| Precision | 1.0000 | 0.8571 |
| Recall | 0.7143 | 0.8571 |
| F1 | 0.8333 | 0.8571 |
| Accuracy | 0.9000 | 0.9000 |

**Key takeaways:**

- v2 catches one more harmful output (`out_015`: the `'5'` response to 2+2) by flagging very short responses.
- v2 introduces one false positive — a trade-off, but for safety filtering, it is worth it.
- Both achieve 90% accuracy, but their precision/recall profiles differ meaningfully.
- **For safety use cases, v2 is better**: higher recall means fewer harmful outputs slip through.
- Accuracy alone would have made them look identical.

**Next up:** Notebook 4 — Mini Project: Full Classifier Pipeline